# Task 5 — Attention-Enhanced GAN

**Goal:** Improve on Task 2's Conditional GAN by adding a **self-attention**
layer, following the SAGAN (Self-Attention GAN, Zhang et al. 2018) approach.
Self-attention lets the Generator relate distant spatial regions of the image
to each other (e.g. making sure petals on opposite sides of a flower are
consistent), which plain convolutions struggle with since they only see local
neighborhoods.

**Why self-attention and not cross-attention:** we're still conditioning on a
class label here (not free-form text yet — that's Task 6). Cross-attention
specifically relates two different sequences (e.g. text tokens attending to
image patches), which we'll use in Task 6. Self-attention, which relates
positions *within* the same image to each other, is the right tool here.

**Controlled comparison with Task 2:** everything else — dataset, label
conditioning mechanism, image size, training setup — is kept identical to
Task 2's CGAN. The only change is the added self-attention block. This makes
it possible to fairly compare "CGAN vs CGAN+attention" results.

## 1. Setup

In [ ]:
!pip install -q torch torchvision matplotlib

import os
import shutil
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as T
from torchvision.datasets import Flowers102
from torchvision.utils import make_grid, save_image
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

## 2. Load Oxford-102 (identical setup to Task 2)

In [ ]:
DATA_ROOT = '/content/data'
os.makedirs(DATA_ROOT, exist_ok=True)

IMG_SIZE = 64
NUM_CLASSES = 102

transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize([0.5]*3, [0.5]*3),
])

train_set = Flowers102(root=DATA_ROOT, split='train', download=True, transform=transform)
val_set   = Flowers102(root=DATA_ROOT, split='val', download=True, transform=transform)
full_train = torch.utils.data.ConcatDataset([train_set, val_set])
print(f'Total training images: {len(full_train)}')

BATCH_SIZE = 64
dataloader = DataLoader(full_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)

## 3. Self-Attention module

This is the new piece. Implements the standard SAGAN self-attention block:
- Projects the feature map into Query, Key, Value via 1x1 convolutions
- Computes attention weights (how much every spatial position should attend
  to every other position)
- Combines attended values back with the original features via a learnable
  residual weight `gamma` (starts at 0, so training starts identical to a
  plain conv network and gradually learns how much attention to use)

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.query = nn.Conv2d(in_channels, in_channels // 8, kernel_size=1)
        self.key   = nn.Conv2d(in_channels, in_channels // 8, kernel_size=1)
        self.value = nn.Conv2d(in_channels, in_channels, kernel_size=1)
        self.gamma = nn.Parameter(torch.zeros(1))  # learnable, starts at 0
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x):
        B, C, H, W = x.size()
        N = H * W

        q = self.query(x).view(B, -1, N).permute(0, 2, 1)   # B x N x C/8
        k = self.key(x).view(B, -1, N)                       # B x C/8 x N
        v = self.value(x).view(B, -1, N)                     # B x C x N

        attn = self.softmax(torch.bmm(q, k))                  # B x N x N  (attention map)
        out = torch.bmm(v, attn.permute(0, 2, 1))             # B x C x N
        out = out.view(B, C, H, W)

        out = self.gamma * out + x   # residual connection
        return out, attn


# Quick shape sanity check
test_attn = SelfAttention(128)
test_input = torch.randn(2, 128, 16, 16)
test_out, test_map = test_attn(test_input)
print('Self-attention output shape:', test_out.shape)
print('Attention map shape:', test_map.shape, '(N x N over the 16x16=256 spatial positions)')

## 4. Generator and Discriminator with self-attention

Identical architecture to Task 2, with one `SelfAttention` block inserted at
the 16x16 feature-map stage in both networks — the standard SAGAN placement
(cheap enough at this resolution, but still expressive spatially).

In [ ]:
LATENT_DIM = 100
LABEL_EMBED_DIM = 50

class AttnGenerator(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, num_classes=NUM_CLASSES, embed_dim=LABEL_EMBED_DIM, img_size=IMG_SIZE):
        super().__init__()
        self.label_embed = nn.Embedding(num_classes, embed_dim)
        input_dim = latent_dim + embed_dim
        self.init_size = img_size // 16  # 4
        self.fc = nn.Linear(input_dim, 256 * self.init_size * self.init_size)

        self.block1 = nn.Sequential(
            nn.BatchNorm2d(256),
            nn.Upsample(scale_factor=2),                    # 4 -> 8
            nn.Conv2d(256, 128, 3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.block2 = nn.Sequential(
            nn.Upsample(scale_factor=2),                    # 8 -> 16
            nn.Conv2d(128, 64, 3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.attn = SelfAttention(64)                       # <-- attention at 16x16, 64 channels
        self.block3 = nn.Sequential(
            nn.Upsample(scale_factor=2),                    # 16 -> 32
            nn.Conv2d(64, 32, 3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Upsample(scale_factor=2),                    # 32 -> 64
            nn.Conv2d(32, 3, 3, stride=1, padding=1),
            nn.Tanh(),
        )

    def forward(self, noise, labels, return_attention=False):
        label_emb = self.label_embed(labels)
        x = torch.cat([noise, label_emb], dim=1)
        x = self.fc(x)
        x = x.view(x.size(0), 256, self.init_size, self.init_size)
        x = self.block1(x)
        x = self.block2(x)
        x, attn_map = self.attn(x)
        img = self.block3(x)
        if return_attention:
            return img, attn_map
        return img


class AttnDiscriminator(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, img_size=IMG_SIZE):
        super().__init__()
        self.img_size = img_size
        self.label_embed = nn.Embedding(num_classes, img_size * img_size)

        self.block1 = nn.Sequential(
            nn.Conv2d(4, 32, 4, stride=2, padding=1),        # 64 -> 32
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, 4, stride=2, padding=1),       # 32 -> 16
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.attn = SelfAttention(64)                        # <-- attention at 16x16, matches Generator
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, 4, stride=2, padding=1),      # 16 -> 8
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(128, 256, 4, stride=2, padding=1),     # 8 -> 4
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.adv_layer = nn.Sequential(
            nn.Linear(256 * (img_size // 16) ** 2, 1),
            nn.Sigmoid()
        )

    def forward(self, img, labels):
        label_map = self.label_embed(labels).view(-1, 1, self.img_size, self.img_size)
        x = torch.cat([img, label_map], dim=1)
        x = self.block1(x)
        x = self.block2(x)
        x, _ = self.attn(x)
        x = self.block3(x)
        x = x.view(x.size(0), -1)
        return self.adv_layer(x)


generator = AttnGenerator().to(device)
discriminator = AttnDiscriminator().to(device)

n_params_g = sum(p.numel() for p in generator.parameters())
n_params_d = sum(p.numel() for p in discriminator.parameters())
print(f'Generator params: {n_params_g:,}')
print(f'Discriminator params: {n_params_d:,}')

## 5. Training setup (identical hyperparameters to Task 2)

In [ ]:
adversarial_loss = nn.BCELoss()

lr = 0.0002
beta1 = 0.5
optimizer_G = optim.Adam(generator.parameters(), lr=lr, betas=(beta1, 0.999))
optimizer_D = optim.Adam(discriminator.parameters(), lr=lr, betas=(beta1, 0.999))

N_EPOCHS = 100
SAMPLE_EVERY = 10

os.makedirs('/content/outputs', exist_ok=True)
os.makedirs('/content/checkpoints', exist_ok=True)

# Same fixed classes as Task 2, so the two experiments are directly comparable
FIXED_CLASSES = [0, 25, 50, 72, 90, 101]
n_fixed = len(FIXED_CLASSES)
fixed_noise = torch.randn(n_fixed, LATENT_DIM, device=device)
fixed_labels = torch.tensor(FIXED_CLASSES, device=device)

## 6. Training loop

Same adversarial training procedure as Task 2. The only difference in the
loop itself is that the Generator/Discriminator now internally use attention
— no change needed to the training logic.

**Time estimate:** slightly slower per epoch than Task 2 due to the added
attention computation (roughly 10–20% more time), so budget ~35–55 minutes
for 100 epochs on a T4.

In [ ]:
g_losses, d_losses = [], []

for epoch in range(1, N_EPOCHS + 1):
    epoch_g_loss, epoch_d_loss = 0.0, 0.0

    for real_imgs, labels in dataloader:
        real_imgs = real_imgs.to(device)
        labels = labels.to(device)
        bs = real_imgs.size(0)

        valid = torch.ones(bs, 1, device=device)
        fake = torch.zeros(bs, 1, device=device)

        # ---- Train Generator ----
        optimizer_G.zero_grad()
        noise = torch.randn(bs, LATENT_DIM, device=device)
        gen_labels = torch.randint(0, NUM_CLASSES, (bs,), device=device)
        gen_imgs = generator(noise, gen_labels)
        g_loss = adversarial_loss(discriminator(gen_imgs, gen_labels), valid)
        g_loss.backward()
        optimizer_G.step()

        # ---- Train Discriminator ----
        optimizer_D.zero_grad()
        real_loss = adversarial_loss(discriminator(real_imgs, labels), valid)
        fake_loss = adversarial_loss(discriminator(gen_imgs.detach(), gen_labels), fake)
        d_loss = (real_loss + fake_loss) / 2
        d_loss.backward()
        optimizer_D.step()

        epoch_g_loss += g_loss.item()
        epoch_d_loss += d_loss.item()

    avg_g = epoch_g_loss / len(dataloader)
    avg_d = epoch_d_loss / len(dataloader)
    g_losses.append(avg_g)
    d_losses.append(avg_d)

    print(f'Epoch [{epoch}/{N_EPOCHS}]  D_loss: {avg_d:.4f}  G_loss: {avg_g:.4f}')

    if epoch % SAMPLE_EVERY == 0 or epoch == 1:
        generator.eval()
        with torch.no_grad():
            samples = generator(fixed_noise, fixed_labels)
        grid = make_grid(samples, nrow=n_fixed, normalize=True)
        save_image(grid, f'/content/outputs/epoch_{epoch:03d}.png')
        generator.train()

torch.save(generator.state_dict(), '/content/checkpoints/attn_generator.pth')
torch.save(discriminator.state_dict(), '/content/checkpoints/attn_discriminator.pth')
print('\nTraining complete. Checkpoints and sample grids saved locally.')

## 7. Plot training curves

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(g_losses, label='Generator loss')
plt.plot(d_losses, label='Discriminator loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Attention-GAN Training Losses')
plt.legend()
plt.tight_layout()
plt.savefig('/content/outputs/training_curves.png', dpi=150)
plt.show()

## 8. Compare early vs late samples

In [ ]:
from PIL import Image as PILImage

early_path = '/content/outputs/epoch_001.png'
late_path = f'/content/outputs/epoch_{N_EPOCHS:03d}.png' if N_EPOCHS % SAMPLE_EVERY == 0 else None

fig, axes = plt.subplots(2, 1, figsize=(12, 6))
axes[0].imshow(PILImage.open(early_path))
axes[0].set_title('Epoch 1 (early training)')
axes[0].axis('off')

if late_path and os.path.exists(late_path):
    axes[1].imshow(PILImage.open(late_path))
    axes[1].set_title(f'Epoch {N_EPOCHS} (final, with self-attention)')
    axes[1].axis('off')

plt.tight_layout()
plt.savefig('/content/outputs/early_vs_late_comparison.png', dpi=150)
plt.show()

## 9. Visualize the attention map itself

This is unique to Task 5 vs Task 2: we can actually look at *what the
Generator's attention layer is attending to*. We overlay the attention
received by the center spatial position onto a generated image, as one
illustrative example of what self-attention is doing internally.

In [ ]:
generator.eval()
with torch.no_grad():
    sample_noise = torch.randn(1, LATENT_DIM, device=device)
    sample_label = torch.tensor([72], device=device)
    img, attn_map = generator(sample_noise, sample_label, return_attention=True)

# attn_map shape: (1, N, N) where N = 16*16 = 256 spatial positions at this layer
N = attn_map.shape[-1]
side = int(N ** 0.5)  # 16
center_idx = (side // 2) * side + (side // 2)  # roughly the center position
attn_from_center = attn_map[0, center_idx].view(side, side).cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
generated_img = (img[0].detach().cpu().permute(1, 2, 0).numpy() + 1) / 2  # unnormalize to [0,1]
axes[0].imshow(generated_img)
axes[0].set_title('Generated image (class 72)')
axes[0].axis('off')

axes[1].imshow(attn_from_center, cmap='inferno')
axes[1].set_title('Attention FROM center position\n(16x16 feature map resolution)')
axes[1].axis('off')

plt.tight_layout()
plt.savefig('/content/outputs/attention_map_visualization.png', dpi=150)
plt.show()
generator.train()

## 10. Copy results to Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/elevance-skills/task5_attention_gan'
os.makedirs(DRIVE_DIR, exist_ok=True)

shutil.copytree('/content/outputs', os.path.join(DRIVE_DIR, 'outputs'), dirs_exist_ok=True)
shutil.copytree('/content/checkpoints', os.path.join(DRIVE_DIR, 'checkpoints'), dirs_exist_ok=True)

print('Copied outputs and checkpoints to:', DRIVE_DIR)

## 11. Summary of findings

Fill this in after training, then copy into `NOTES.md` and today's daily log:
- Compare loss curves and sample quality against Task 2's plain CGAN
- Did images look more coherent/structured with attention?
- What did the attention map visualization show — did it attend to
  sensible/interesting regions, or look fairly diffuse/random?
- Was training noticeably slower than Task 2?